<div dir="rtl">

# 05 — إنهاء النماذج وحفظها في `models/`

أول نوتبوك في المشروع يحفظ آثاراً فعلية في `models/` (حتى الآن كل التدريب كان في نوتبوكات تحليلية بلا حفظ).
أربعة مخرجات:

1. `models/logreg_demo.pkl` — `LogisticRegression(max_iter=1000, class_weight="balanced")` على عيّنة طبقية
   200,000 صف (سريع، للتجربة/العرض).
2. `models/logreg_final.pkl` — **نفس الإعداد بالضبط، على `train` الكامل (1,025,602 صف)** — أول تدريب على
   البيانات الكاملة في هذا المشروع.
3. `models/nb_evidence_table.npz` — جدول `P(evidence|disease)` (Bayes الساذج من `04_smart_selection.ipynb`)،
   محفوظاً لإعادة استخدامه دون إعادة بنائه.
4. `models/label_encoder.joblib` — نسخة من `data/processed/label_encoder.joblib` داخل `models/` (بجانب
   النماذج التي تحتاجه). **تنويه**: الاسم الصحيح `.joblib` لا `.pkl` — امتداد الملف الفعلي في المشروع منذ
   `02_encode.ipynb`.

في النهاية: مقارنة `logreg_demo` مقابل `logreg_final` (Accuracy وundertriage_rate عند k=5,8) على نفس
`validate` — هل التدريب على البيانات الكاملة يستحق تكلفته الزمنية (~15 دقيقة مقابل ~5 دقائق)؟

</div>

In [1]:
import ast
import json
import shutil
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
SAMPLE_SIZE = 200_000
K_VALUES = (5, 8)
ALPHA = 1.0  # تنعيم لابلاس لجدول Bayes الساذج

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

<div dir="rtl">

## تحميل البيانات

</div>

In [2]:
with open(DATA_PROCESSED / "feature_names.json", encoding="utf-8") as f:
    feature_names = json.load(f)
label_encoder = joblib.load(DATA_PROCESSED / "label_encoder.joblib")

X_train_full = sparse.load_npz(DATA_PROCESSED / "train_features.npz")
y_train_full = np.load(DATA_PROCESSED / "train_labels.npy")
X_validate = sparse.load_npz(DATA_PROCESSED / "validate_features.npz")
y_validate = np.load(DATA_PROCESSED / "validate_labels.npy")

evidence_columns = feature_names[:-2]
item_to_col_idx = {name: idx for idx, name in enumerate(evidence_columns)}
N_EVID = len(evidence_columns)

with open(DATA_RAW / "release_conditions.json", encoding="utf-8") as f:
    conditions = json.load(f)
severity_by_class = np.array([conditions[c]["severity"] for c in label_encoder.classes_])

print(f"X_train_full: {X_train_full.shape}, X_validate: {X_validate.shape}")

X_train_full: (1025602, 974), X_validate: (132448, 974)


<div dir="rtl">

## 1) `logreg_demo.pkl` — تدريب على عيّنة طبقية 200,000 صف

</div>

In [3]:
X_train_demo, _, y_train_demo, _ = train_test_split(
    X_train_full, y_train_full, train_size=SAMPLE_SIZE, stratify=y_train_full, random_state=RANDOM_STATE
)

t0 = time.time()
model_demo = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
model_demo.fit(X_train_demo, y_train_demo)
print(f"logreg_demo (200,000 صف): تم بعد {time.time() - t0:.1f}s (n_iter={model_demo.n_iter_})")

joblib.dump(model_demo, MODELS / "logreg_demo.pkl")
print(f"حُفظ: {MODELS / 'logreg_demo.pkl'}")

logreg_demo (200,000 صف): تم بعد 287.5s (n_iter=[877])
حُفظ: ../models/logreg_demo.pkl


<div dir="rtl">

## 2) `logreg_final.pkl` — تدريب على `train` الكامل (1,025,602 صف)

أول تدريب في هذا المشروع على البيانات الكاملة بلا أخذ عيّنة — أبطأ (~15 دقيقة) لكنه يستخدم كل البيانات
المتاحة.

</div>

In [4]:
t0 = time.time()
model_final = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
model_final.fit(X_train_full, y_train_full)
print(
    f"logreg_final (train الكامل، {X_train_full.shape[0]:,} صف): "
    f"تم بعد {time.time() - t0:.1f}s (n_iter={model_final.n_iter_})"
)

joblib.dump(model_final, MODELS / "logreg_final.pkl")
print(f"حُفظ: {MODELS / 'logreg_final.pkl'}")

logreg_final (train الكامل، 1,025,602 صف): تم بعد 904.9s (n_iter=[568])
حُفظ: ../models/logreg_final.pkl


<div dir="rtl">

## 3) `nb_evidence_table.npz` — جدول Bayes الساذج (من `04_smart_selection.ipynb`)

نفس البناء بالضبط: نسبة مرضى كل مرض الذين ظهر لديهم كل كود فعلاً في `train` الكامل (تنعيم لابلاس `α=1`)،
محسوبة مباشرة من `X_train_full` المُرمَّز. يُحفَظ مع `class_prior` وقائمة الأكواد والفئات لإعادة الاستخدام
دون إعادة البناء.

</div>

In [5]:
def ecode(item):
    return item.split("_@_")[0]


code_to_colidxs = {}
for idx, name in enumerate(evidence_columns):
    code_to_colidxs.setdefault(ecode(name), []).append(idx)
all_codes = sorted(code_to_colidxs.keys())
code_to_idx = {c: i for i, c in enumerate(all_codes)}
n_codes = len(all_codes)

X_train_full_csc = X_train_full.tocsc()
n_classes_total = len(label_encoder.classes_)
class_counts_full = np.bincount(y_train_full, minlength=n_classes_total).astype(float)
class_prior = class_counts_full / class_counts_full.sum()

P_present = np.zeros((n_classes_total, n_codes))
t0 = time.time()
for c in all_codes:
    cols = code_to_colidxs[c]
    presence = np.asarray((X_train_full_csc[:, cols].sum(axis=1) > 0)).ravel()
    present_counts = np.bincount(y_train_full, weights=presence.astype(float), minlength=n_classes_total)
    P_present[:, code_to_idx[c]] = (present_counts + ALPHA) / (class_counts_full + 2 * ALPHA)
print(f"جدول Bayes الساذج ({P_present.shape}): تم بعد {time.time() - t0:.1f}s")

np.savez(
    MODELS / "nb_evidence_table.npz",
    P_present=P_present,
    class_prior=class_prior,
    all_codes=np.array(all_codes),
    classes=label_encoder.classes_,
)
print(f"حُفظ: {MODELS / 'nb_evidence_table.npz'}")

جدول Bayes الساذج ((49, 223)): تم بعد 2.5s
حُفظ: ../models/nb_evidence_table.npz


<div dir="rtl">

## 4) نسخ `label_encoder` إلى `models/`

</div>

In [6]:
src = DATA_PROCESSED / "label_encoder.joblib"
dst = MODELS / "label_encoder.joblib"
shutil.copy(src, dst)
print(f"نُسخ: {src} -> {dst}")

نُسخ: ../data/processed/label_encoder.joblib -> ../models/label_encoder.joblib


<div dir="rtl">

## مقارنة `logreg_demo` مقابل `logreg_final`: Accuracy وundertriage_rate عند k=5,8

نفس منهج الإسقاط العشوائي من `03_train.ipynb` (`INITIAL_EVIDENCE` + k دليل عشوائي، `random_state=42`) على
`validate` الكامل (132,448 صف) — كلا النموذجين يُقيَّمان على نفس الترميز بالضبط عند كل k.

</div>

In [7]:
def encode_evidences(evidences_parsed, item_to_col_idx, n_evidence_cols):
    exploded = evidences_parsed.explode()
    col_idx = exploded.map(item_to_col_idx).to_numpy(dtype=np.int64)
    row_idx = exploded.index.to_numpy()
    data = np.ones(len(exploded), dtype=np.int8)
    return sparse.coo_matrix(
        (data, (row_idx, col_idx)), shape=(len(evidences_parsed), n_evidence_cols)
    ).tocsr()


def build_X_from_kept(kept_items_list, age_arr, sex_arr):
    kept_series = pd.Series(kept_items_list)
    evid = encode_evidences(kept_series, item_to_col_idx, N_EVID)
    age_c = sparse.csr_matrix(np.asarray(age_arr, dtype=float).reshape(-1, 1))
    sex_c = sparse.csr_matrix(np.asarray(sex_arr, dtype=float).reshape(-1, 1))
    return sparse.hstack([evid, age_c, sex_c]).tocsr()


def evaluate_basic(model, X, y_true):
    y_pred = model.predict(X)
    sev_true = severity_by_class[y_true]
    sev_pred = severity_by_class[y_pred]
    undertriage_rate = float(np.mean(sev_pred > sev_true))
    return {"Accuracy": accuracy_score(y_true, y_pred), "undertriage_rate": undertriage_rate}


df_val = pd.read_csv(DATA_RAW / "release_validate_patients.csv")
df_val["EVIDENCES_PARSED"] = df_val["EVIDENCES"].apply(ast.literal_eval)
codes_per_row_full = df_val["EVIDENCES_PARSED"].apply(lambda items: sorted(set(ecode(it) for it in items)))
initial_code_full = df_val["INITIAL_EVIDENCE"].apply(ecode)
items_full = df_val["EVIDENCES_PARSED"]

age_col_full = X_validate[:, N_EVID].toarray().ravel()
sex_col_full = X_validate[:, N_EVID + 1].toarray().ravel()


def make_random_ablated_full(k, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    kept_items_per_row = []
    for items, codes, ic in zip(items_full, codes_per_row_full, initial_code_full):
        remaining = [c for c in codes if c != ic]
        n_take = min(k, len(remaining))
        sampled = (
            rng.choice(np.array(remaining, dtype=object), size=n_take, replace=False).tolist()
            if n_take > 0
            else []
        )
        keep_codes = set(sampled) | {ic}
        kept_items_per_row.append([it for it in items if ecode(it) in keep_codes])
    return kept_items_per_row


comparison_rows = []
for k in K_VALUES:
    kept = make_random_ablated_full(k)
    X_k = build_X_from_kept(kept, age_col_full, sex_col_full)
    for name, m in (("logreg_demo", model_demo), ("logreg_final", model_final)):
        metrics = evaluate_basic(m, X_k, y_validate)
        comparison_rows.append({"k": k, "النموذج": name, **metrics})
        print(f"k={k} {name}: Accuracy={metrics['Accuracy']:.4f} undertriage_rate={metrics['undertriage_rate']:.4f}")

k=5 logreg_demo: Accuracy=0.7855 undertriage_rate=0.0630
k=5 logreg_final: Accuracy=0.8196 undertriage_rate=0.0574
k=8 logreg_demo: Accuracy=0.9273 undertriage_rate=0.0237
k=8 logreg_final: Accuracy=0.9380 undertriage_rate=0.0221


In [8]:
comparison_df = pd.DataFrame(comparison_rows).set_index(["k", "النموذج"])
comparison_df

                Accuracy  undertriage_rate
k النموذج
5 logreg_demo   0.785508          0.062991
  logreg_final  0.819590          0.057426
8 logreg_demo   0.927300          0.023723
  logreg_final  0.937991          0.022129

<div dir="rtl">

## ملخص وخارج النطاق

- **`logreg_final` (train الكامل) يتفوَّق باستمرار على `logreg_demo` (عيّنة 200 ألف)** على كلا المقياسين
  وعند كلا k: Accuracy أعلى بـ+3.4 نقطة عند k=5 (82.0% مقابل 78.6%) وبـ+1.1 نقطة عند k=8 (93.8% مقابل 92.7%)؛
  `undertriage_rate` أقل (أفضل) في كل الحالات. **الاستنتاج**: استخدام البيانات الكاملة يستحق تكلفته الزمنية
  (~15 دقيقة) — تحسّن حقيقي وليس ضجيجاً، خصوصاً عند نقص الأدلة (k=5) حيث الفارق أكبر.
- ملاحظة تقنية: `logreg_final` تقارب بعدد تكرارات أقل (`n_iter=568`) من `logreg_demo` (`n_iter=877`) رغم
  حجم بيانات أكبر بـ5.1 مرة — تقديرات تدرّج أنظف مع بيانات أكثر تُسرِّع التقارب لكل تكرار، حتى لو كانت كل
  تكرارة أغلى حسابياً.
- **أربعة ملفات محفوظة فعلياً في `models/`**: `logreg_demo.pkl`، `logreg_final.pkl`، `nb_evidence_table.npz`،
  `label_encoder.joblib`. `.pkl` و`.joblib` مستثناة من git مسبقاً (`models/*.pkl`, `models/*.joblib` في
  `.gitignore`)؛ `models/*.npz` أُضيفت الآن لنفس السبب (لا رفع آثار نموذج محلية لـgit).
- `test` لم يُلمس. لا XGBoost.

</div>